# Device [DEVICE_ID] Analysis

**Purpose:** [one-line description of what you're investigating]  
**Date started:** [date]  
**Data subset:** [e.g., row_number 51-425, repeats=1M]

---
## Setup

In [7]:
# === Standard imports (don't modify) ===
import sys
from pathlib import Path
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

# Project setup
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.parquet_data_manager import ParquetExperimentalDataManager
from src.analysis import (
    find_p_ap_states,
    calculate_switching_probability,
    visualize_peaks_polars,
    xorn_whitening,
)
from src.analysis.visualization import (
    plot_binomial_histogram,
    plot_switching_scatter,
    plot_comparison_subplots,
)
from src.analysis.stats import get_bernoulli_trials_statistics

# Initialize data manager
db_path=project_root / "database" / "mtj_switching_data_parquet.db"
print(f"Database path: {db_path}")
print(f"Database exists: {db_path.exists()}")

data_manager = ParquetExperimentalDataManager(
    db_path=project_root / "database" / "mtj_switching_data_parquet.db",
    parquet_dir=project_root / "database" / "parquet_data"
)

2026-02-25 11:25:40,206 - INFO - Initializing database...


Database path: /Users/shannonegan/Documents/DSV/StochastIQ/umn/mtj-trng-analysis/database/mtj_switching_data_parquet.db
Database exists: True


In [8]:
# === Plot style config (modify these, not individual cells) ===
#
# All figure formatting lives HERE. When you tweak fontsize or width
# in a scratch cell, move the final values back up to this cell.

PLOT_STYLE = {
    'width_cm': 13,
    'aspect': 0.8,
    'fontsize': 16,
    'linewidth': 3,
    'dpi': 300,
}

_CM_TO_INCH = 1 / 2.54

def figsize(width_cm=None, aspect=None):
    """Get figsize tuple from cm width and aspect ratio."""
    w = (width_cm or PLOT_STYLE['width_cm']) * _CM_TO_INCH
    a = aspect or PLOT_STYLE['aspect']
    return (w, a * w)

def apply_style():
    """Apply the global plot style. Call once, or again after plt.rcdefaults()."""
    plt.rcParams.update({
        'font.family': 'Times New Roman',
        'mathtext.fontset': 'cm',
        'font.size': PLOT_STYLE['fontsize'],
        'font.weight': 'bold',
        'axes.labelweight': 'bold',
    })

apply_style()

# Output paths
figpath = project_root / 'sample_plots' / 'device_011_08'  # <-- update
figpath.mkdir(parents=True, exist_ok=True)

In [ ]:
# === Load data for this device ===
device_id = '011_08'  # <-- update per notebook

# Load run metadata
import sqlite3

with sqlite3.connect(data_manager.db_path) as conn:
    runs_df = pl.read_database('SELECT * FROM runs', conn)
print(runs_df.columns)

device_df = runs_df.filter(pl.col('device_id') == device_id)
print(f'Loaded {len(device_df)} runs for device {device_id}')

# Show parameter groups
device_df.select(
    ['row_number', 's_amplitude', 's_width', 'r_amplitude', 'r_width', 'field', 'repeats', 'switching_probability']
).sort('row_number').group_by(
    ['s_amplitude', 's_width', 'r_amplitude', 'r_width', 'field', 'repeats'],
    maintain_order=True
).agg([
    pl.col('row_number').min().alias('first_row'),
    pl.col('row_number').max().alias('last_row'),
    pl.len().alias('n_runs'),
])

['run_id', 'run_hash', 'device_id', 'filename', 'hz', 'row_number', 'r_amplitude', 'r_width', 's_amplitude', 's_width', 'field', 'source', 'repeats', 'import_timestamp', 'parquet_file', 'binary_parquet_column', 'switching_probability', 'device_id_fk', 'parquet_column']
Loaded 423 runs for device 011_08


s_amplitude,s_width,r_amplitude,r_width,field,repeats,first_row,last_row,n_runs
f64,f64,f64,f64,f64,i64,i64,i64,u32
5.96,5.0000e-10,-0.6,0.0000001,-800.0,10000,1,169,3
5.96,1.0000e-9,-0.6,0.0000001,-800.0,10000,2,170,3
5.96,1.5000e-9,-0.6,0.0000001,-800.0,10000,3,171,3
5.96,2.0000e-9,-0.6,0.0000001,-800.0,10000,4,172,3
5.96,2.5000e-9,-0.6,0.0000001,-800.0,10000,5,173,3
…,…,…,…,…,…,…,…,…
5.96,5.0000e-9,-0.8,0.0000001,-800.0,1000000,360,377,18
5.96,6.0000e-9,-0.8,0.0000001,-800.0,1000000,378,393,16
5.96,7.0000e-9,-0.8,0.0000001,-800.0,1000000,394,409,16


In [ ]:
# === Select analysis subset ===
# Change this filter for each analysis pass

analysis_filter = (
    (pl.col("row_number").is_between(243, 260))
    & (pl.col('repeats') == 1_000_000)
)

analysis_df = device_df.filter(analysis_filter).sort('row_number')
run_ids = analysis_df['run_id'].to_list()
print(f'Selected {len(run_ids)} runs')

# Load measurements
measurements = data_manager.get_measurements(run_ids)
stacked = data_manager.stack_measurements(measurements)
print(f'Stacked shape: {stacked.shape}')

# Get device threshold
device_info = data_manager.get_device_info(device_id)
if device_info:
    threshold = device_info[1]
    peaks = [device_info[2], device_info[3]]
    print(f'Threshold: {threshold:.6f}, P: {peaks[0]:.6f}, AP: {peaks[1]:.6f}')
else:
    threshold, peaks, tmr = find_p_ap_states(stacked)
    print(f'Calculated threshold: {threshold:.6f}')

Selected 18 runs
Stacked shape: (18000000, 3)
Threshold: 0.009735, P: 0.007396, AP: 0.012074


---
## Scratch

Exploratory work goes below. Messy is fine. When something works well,
clean it up and move it to the **Promote** section at the bottom.

In [ ]:
# Scratch cell — explore here


---
## Promote

Functions and plots that are worth reusing go here.  
**Rule of thumb:** if you copy-paste it to another notebook, move it to `src/` instead.

In [ ]:
# Promoted functions — staging area before src/
